## Provide overview of results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Optional, Tuple
from my_utils import LABELING_FUNCTION_2_PAPER_NAME

import glob
import os

import sys
sys.path.append("../UKB_validation/")
#import python file containing filenames
import filepaths as fp

# import functions from Scripts/plot.py and Scripts/8_make_results_plots.py
from Scripts.make_results_plots import plot_all_labeling_functions, plot_all_task_groups, plot_all_task_group_box_plots, plot_radar_chart, plot_sensitivity_analysis, plot_comparison
from my_utils import TASK_GROUP_2_LABELING_FUNCTION
from Scripts.plot import filter_df

In [ ]:
#phecodes of interest (change if wanted)
phecodes_disease_onset = ['phecode_RE_468', 'phecode_CV_404.1', 'phecode_MS_705.1', 'phecode_MS_718', 'phecode_RE_474', 'phecode_CV_431.11', 'phecode_CV_440.3', 'phecode_EM_202', 'phecode_GU_582.2', 'phecode_MB_284', 'phecode_CV_410.2', 'phecode_CV_413.11', 'phecode_CV_438.11', 'phecode_DE_664.4', 'phecode_CV_404', 'phecode_NS_324.11', 'phecode_CV_400', 'phecode_CV_413.21', 'phecode_CV_416.21', 'phecode_CV_420', 'phecode_BI_164', 'phecode_CV_401', 'phecode_CV_424']
disease_name_disease_onset = [
    "Pneumonia",
    "Myocardial infarction [Heart attack]",
    "Rheumatoid arthritis",
    "Back pain",
    "Chronic obstructive pulmonary disease [COPD]",
    "Cerebral infarction [Ischemic stroke]",
    "Pulmonary embolism",
    "Diabetes mellitus",
    "Chronic kidney disease",
    "Suicide ideation and attempt or self harm",
    "Endocarditis",
    "Mitral valve insufficiency",
    "Abdominal aortic aneurysm",
    "Psoriasis",
    "Ischemic heart disease",
    "Parkinson's disease (Primary)",
    "Rheumatic fever and chronic rheumatic heart diseases",
    "Aortic stenosis",
    "Atrial fibrillation",
    "Cardiac arrest",
    "Anemia",
    "Hypertension",
    "Heart failure",
]

In [ ]:
indications = dict(
    disease_onset = disease_name_disease_onset,
    hospitalization = ["hospitalization"],
    death = ["death"]
)
method = "auroc" # auprc, auroc, brier


In [ ]:
indications

In [ ]:
all_scores = []

# Calculate global min and max for y-axis
global_min = float('inf')
global_max = float('-inf')

num_cv_rounds = 5

# Loop through diseases and cases to load CSVs and find min/max
for indication, phecodelist in indications.items():
    for disease in phecodelist:
            scores_filename = f"{fp.BASE_PATH_RESULTS}Results_Revision3/{glob.escape(disease)}_all_results.csv"
            matching_files = glob.glob(scores_filename)
            
            if(len(matching_files) == 0):
                print(f"Skipping {disease}")
                continue
            # Load the CSV#
            scores = pd.read_csv(matching_files[0], index_col=0)
            all_scores.append(scores)

all_scores = pd.concat(all_scores)

## rename sub_task - name everything disease_onset except hospitalization and death
all_scores['sub_task'] = all_scores['sub_task'].apply(lambda x: 'disease_onset' if x not in ['hospitalization', 'death'] else 'OMOP_9201' if x == 'hospitalization' else 'OMOP_4306655')

#only keeps rows where crossval_iteration is between 0 and 4
all_scores

In [ ]:
## filter all_scores based on the following criteria:
max_k = {
    'Cardiac arrest': 32,
    'Abdominal aortic aneurysm': 24,
    'Aortic stenosis': 48,
    'Mitral valve insufficiency': 64,
    'Endocarditis': 24,
    'Rheumatic fever and chronic rheumatic heart diseases': 64,
    "Parkinson's disease (Primary)": 32,
    'Suicide ideation and attempt or self harm': 64,
}

limits = all_scores['labeling_function'].map(max_k)

all_scores = all_scores[
    limits.isna() | (all_scores['k'] <= limits)
]


In [ ]:
all_scores["labeling_function"].nunique()

In [ ]:
PATH_TO_OUTPUT_DIR = "./plots/"
os.makedirs(PATH_TO_OUTPUT_DIR, exist_ok=True)

MODEL_HEADS = None  
all_scores_filtered = all_scores[all_scores["model"].isin(["count", "clmbr", "qwen3", "bioclinicalbert"])]

# Individual task plots
plot_all_labeling_functions(
    all_scores_filtered[all_scores_filtered["sub_task"] == "disease_onset"],
    score=method,
    path_to_output_dir=PATH_TO_OUTPUT_DIR,
    model_heads=MODEL_HEADS,
    is_x_scale_log=True,
    is_std_bars=True
)

# Task group aggregated plots
plot_all_task_groups(
    all_scores_filtered,
    score=method,
    path_to_output_dir=PATH_TO_OUTPUT_DIR,
    model_heads=MODEL_HEADS,
    is_x_scale_log=True,
    shadedregion=True,
    addlinesdiagnoses=True
)

# Boxplots
plot_all_task_group_box_plots(
    all_scores_filtered,
    score=method,
    path_to_output_dir=PATH_TO_OUTPUT_DIR,
    model_heads=MODEL_HEADS
)

# Radar plot for specific k
plot_radar_chart(
    all_scores_filtered,
    k=32,
    path_to_output_dir=PATH_TO_OUTPUT_DIR,
    score=method,
    model_heads=MODEL_HEADS
)

In [ ]:
model_name = {
    "qwen": "GTE-Qwen2-7B+LR",
    "qwen3": "Qwen3-Emb-8B+LR",
    "llm2vec": "LLM2Vec-Llama-3.1 8B+LR",
    "clmbr": "CLMBR-T-Base+LR",
    "count": "Count-based+GBM",
    "bioclinicalbert": "BioClinicalBERT+LR",
    "qwenclmbrcodes": "Sensitivity analysis"
}

model_palette = {
    "Qwen3-Emb-8B+LR": "#1f77b4",     # blue
    "CLMBR-T-Base+LR": "#ff7f0e",                  # orange
    "Count-based+GBM": "#2ca02c",          # green
    "LLM2Vec-Llama-3.1 8B+LR": "#d62728",   # red
    "GTE-Qwen2-7B+LR": "#9467bd",           # purple
    "BioClinicalBERT+LR": "#8c564b",         # brown
    "Sensitivity analysis": "#e377c2"        # pink
}

model_markers = {
    "Qwen3-Emb-8B+LR": "X",
    "CLMBR-T-Base+LR": "o",
    "Count-based+GBM": "p",
    #"LLM2Vec-Llama-3.1 8B+LR": "^", 
    #"GTE-Qwen2-7B+LR": "*",
    "BioClinicalBERT+LR": "^",
    "Qwen3-Emb-8B CLMBR-T-Base codes+LR": "^",
    "Sensitivity analysis": "p"
}

task_mapping = {
    "disease_onset": "Assignment of New Diagnoses",
    "OMOP_9201": "Operational Outcomes",
    "OMOP_4306655": "Mortality Prediction",
}

selected_diseases = {"OMOP_4306655", "OMOP_9201"}

legend_order = ["Qwen3-Emb-8B+LR", "CLMBR-T-Base+LR", "Count-based+GBM", "LLM2Vec-Llama-3.1 8B+LR", "GTE-Qwen2-7B+LR", "BioClinicalBERT+LR", "Sensitivity analysis"]


In [ ]:
def create_table(subtasks, valuetable):
    # -------------------------------------------------
    # 1. Filter for AUROC and specific k
    # -------------------------------------------------
    k_value = -1   # choose 2 or 4
    if subtasks:
        relcol = "sub_task"
    else:
        relcol = "labeling_function"

    filtered_scores = (
        all_scores[
            (all_scores["score"] == "auroc") &
            (all_scores["k"] == k_value)
        ]
        .copy()
    )

    # Optional: rename models
    filtered_scores["model"] = filtered_scores["model"].map(model_name)

    # -------------------------------------------------
    # 2. Aggregate over replicates
    # -------------------------------------------------
    agg = (
        filtered_scores
        .groupby(["model", relcol])
        .agg(
            mean=("mean", "mean"),
            count=("mean", "count"),
            var_sum=("std", lambda x: np.sum(x**2)),
        )
        .reset_index()
    )

    # -------------------------------------------------
    # 3. Add std
    # -------------------------------------------------
    agg["std"] = np.sqrt(agg["var_sum"] / (agg["count"] ** 2))
    agg["lower"] = agg["mean"] - 1.96 * agg["std"]
    agg["upper"] = agg["mean"] + 1.96 * agg["std"]

    # -------------------------------------------------
    # 4. Create formatted string
    # -------------------------------------------------
    agg["auroc_formatted"] = agg.apply(
        lambda row: f"{row['mean']:.3f} ({row['lower']:.3f}–{row['upper']:.3f})",
        axis=1
    )

    # -------------------------------------------------
    # 5. Order models
    # -------------------------------------------------
    model_order = list(model_name.values())
    agg["model"] = pd.Categorical(
        agg["model"],
        categories=model_order,
        ordered=True
    )

    # -------------------------------------------------
    # 6. Pivot table (models x labeling_functions)
    # -------------------------------------------------
    pretty_table = agg.pivot(
        index="model",
        columns=relcol,
        values="auroc_formatted"
    )

    # -------------------------------------------------
    # 7. Mean across labeling_functions
    # -------------------------------------------------
    numerical_pivot = agg.pivot(
        index="model",
        columns=relcol,
        values="mean"
    )

        # get per-LF stds (NOT recomputed from means)
    std_pivot = agg.pivot(
        index="model",
        columns=relcol,
        values="std"
    )

    mean_across = numerical_pivot.mean(axis=1)
    n = numerical_pivot.shape[1]

    # sum variances like original code
    var_sum = (std_pivot**2).sum(axis=1)

    std_total = np.sqrt(var_sum / (n**2))

    lower = mean_across - 1.96 * std_total
    upper = mean_across + 1.96 * std_total

    mean_across = mean_across.round(3)
    lower = lower.round(3)
    upper = upper.round(3)

    mean_formatted = pd.DataFrame({
        f"Mean Across LFs (k={k_value})":
            [f"{m:.3f} ({l:.3f}–{u:.3f})"
            for m, l, u in zip(mean_across, lower, upper)]
    }, index=mean_across.index)

    pretty_table = pretty_table.join(mean_formatted)

    # Ensure correct order
    pretty_table = pretty_table.reindex(model_order)

    return pretty_table

all_scores_filtered_allmodels = all_scores[all_scores["model"].isin(["count", "clmbr", "qwen3", "bioclinicalbert", "llm2vec", "qwen", "qwenclmbrcodes"])]
pretty_table_subtasks = create_table(subtasks=True, valuetable=all_scores_filtered_allmodels)
pretty_table_subtasks

In [ ]:
pretty_table_subtasks.drop(["GTE-Qwen2-7B+LR", "LLM2Vec-Llama-3.1 8B+LR"])

## Generate plot showing sensitivity analysis (Clmbr vs ukb clmbr subset using qwen3)

In [ ]:
all_scores_filtered_sensitivity = all_scores[all_scores["model"].isin(["clmbr", "qwen3", "qwenclmbrcodes"])]

In [ ]:
all_scores_filtered_sensitivity

In [ ]:
# Task group aggregated plots
plot_sensitivity_analysis( #plot_all_task_groups
    all_scores_filtered_sensitivity,
    score=method,
    path_to_output_dir=PATH_TO_OUTPUT_DIR,
    model_heads=None,
    is_x_scale_log=True,
    shadedregion=True
)

In [ ]:
all_scores_filtered_sensitivity

In [ ]:
import pandas as pd

cols_to_mean = ["value", "std", "lower", "mean", "upper"]

df_combined = (
    all_scores_filtered_sensitivity.groupby(
        ["sub_task", "model", "head", "replicate", "k", "score"],
        as_index=False
    )[cols_to_mean]
    .mean()
)
df_combined

## Generate plot comparing EHRSHOT and UKB results

In [ ]:
## Read in data from EHRShot task groups
chexpert_df = pd.read_csv(f"{fp.BASE_PATH_FIGS_EHRSHOT}{method}/chexpert_pretty.csv")
lab_values_df = pd.read_csv(f"{fp.BASE_PATH_FIGS_EHRSHOT}{method}/lab_values_pretty.csv")
new_diagnoses_df = pd.read_csv(f"{fp.BASE_PATH_FIGS_EHRSHOT}{method}/new_diagnoses_pretty.csv")
operational_outcomes_df = pd.read_csv(f"{fp.BASE_PATH_FIGS_EHRSHOT}{method}/operational_outcomes_pretty.csv")
operational_outcomes_df

In [ ]:
all_dfs_ehrshot = pd.DataFrame()
for df, subtask in [(chexpert_df, "chexpert"), (lab_values_df, "lab_values"), (operational_outcomes_df, "operational_outcomes"), (new_diagnoses_df, "new_diagnoses")]:
    score = method
    df_long = (
        df.melt(
            id_vars=["model", "head"],
            var_name="k",
            value_name="value"
        )
    )

    # convert "All" -> -1 and cast to int
    df_long["k"] = df_long["k"].replace({"All": -1}).astype(int)
    df_long["sub_task"] = subtask
    df_long["score"] = method
    df_long["labeling_function"] = "EHRSHOT"
    all_dfs_ehrshot = pd.concat([all_dfs_ehrshot, df_long], ignore_index=True)

all_dfs_ehrshot = all_dfs_ehrshot[["sub_task", "model", "head", "k", "score", "value", "labeling_function"]]
# rename model llm-bert to bioclinicalbert and llm to qwen3
all_dfs_ehrshot["model"] = all_dfs_ehrshot["model"].replace({"llm-bert": "bioclinicalbert", "llm": "qwen3"})
all_dfs_ehrshot

In [ ]:
# get all relevant data from UKB and combine disease_onset values
task_groups = ['Mortality Prediction', 'Operational Outcomes', 'Assignment of New Diagnoses']
combined_dfs = pd.DataFrame()
for task in task_groups:
    df_filtered = filter_df(all_scores, score=score, task_group=task, model_heads=None)
    combined_dfs = pd.concat([combined_dfs, df_filtered], axis=0)

cols_to_mean = ["value", "std", "lower", "mean", "upper"]
df_combined = (
    combined_dfs.groupby(
        ["sub_task", "model", "head", "replicate", "k", "score"],
        as_index=False
    )[cols_to_mean]
    .mean()
)
df_combined["labeling_function"] = df_combined["sub_task"]
all_dfs_ukb = df_combined[["sub_task", "model", "head", "k", "score", "value"]]
all_dfs_ukb["labeling_function"] = "UKB"
all_dfs_ukb = all_dfs_ukb[all_dfs_ukb["model"] != "qwenclmbrcodes"]
all_dfs_ukb.head()

In [ ]:
## combine the two dataframes and create plots with new function
all_dfs_combined = pd.concat([all_dfs_ukb, all_dfs_ehrshot], ignore_index=True)

In [ ]:
# Task group aggregated plots
plot_comparison( #plot_all_task_groups
    all_dfs_combined[~all_dfs_combined["model"].isin(["qwen", "llm2vec"])],
    score=method,
    path_to_output_dir=PATH_TO_OUTPUT_DIR,
    model_heads=None,
    is_x_scale_log=True,
    shadedregion=False
)